In [1]:
import pandas as pd

df = pd.read_csv("data-raw/DataCoSupplyChainDataset.csv", encoding="latin-1")
print(df.shape)

(180519, 53)


In [2]:
drop_cols = ["Customer Email", "Customer Password", "Customer Fname", "Customer Lname",
             "Customer Street", "Customer Zipcode", "Product Description", "Product Image",
             "Order Zipcode", "Latitude", "Longitude"]
df = df.drop(columns=drop_cols)

df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[()]", "", regex=True)
              .str.replace(r"\s+", "_", regex=True))
df = df.rename(columns={"order_date_dateorders": "order_date",
                        "shipping_date_dateorders": "shipping_date",
                        "type": "payment_type"})
print(df.shape)
print(list(df.columns))

(180519, 42)
['payment_type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'department_id', 'department_name', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_name', 'product_price', 'product_status', 'shipping_date', 'shipping_mode']


In [3]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["shipping_date"] = pd.to_datetime(df["shipping_date"])
df["delay_days"] = df["days_for_shipping_real"] - df["days_for_shipment_scheduled"]
df["is_cancelled"] = df["order_status"].isin(["CANCELED", "SUSPECTED_FRAUD"])
df[["order_date", "shipping_date", "days_for_shipping_real", "days_for_shipment_scheduled", "delay_days", "is_cancelled"]].head()

,order_date,shipping_date,days_for_shipping_real,days_for_shipment_scheduled,delay_days,is_cancelled
0,2018-01-31 22:56:00,2018-02-03 22:56:00,3,4,-1,False
1,2018-01-13 12:27:00,2018-01-18 12:27:00,5,4,1,False
2,2018-01-13 12:06:00,2018-01-17 12:06:00,4,4,0,False
3,2018-01-13 11:45:00,2018-01-16 11:45:00,3,4,-1,False
4,2018-01-13 11:24:00,2018-01-15 11:24:00,2,4,-2,False


In [12]:
results = []
def check(name, passed, detail=""):
    results.append({"check": name, "result": "PASS" if passed else "FAIL", "detail": detail})

check("order_item_id is unique", df["order_item_id"].is_unique)

key_cols = ["order_item_id", "order_id", "order_customer_id", "product_card_id", "order_date", "shipping_mode", "market"]
n = int(df[key_cols].isna().sum().sum())
check("no nulls in key columns", n == 0, f"{n} nulls")

n = int((((df["days_for_shipping_real"] > df["days_for_shipment_scheduled"]).astype(int) != df["late_delivery_risk"]) & ~df["is_cancelled"]).sum())
check("late_delivery_risk = real > scheduled (non-cancelled orders)", n == 0, f"{n} rows differ")

n = int((df.loc[df["is_cancelled"], "late_delivery_risk"] != 0).sum())
check("cancelled orders have late_delivery_risk = 0", n == 0, f"{n} rows")

n = int((df["delivery_status"].eq("Shipping canceled") != df["is_cancelled"]).sum())
check("'Shipping canceled' = order CANCELED or SUSPECTED_FRAUD", n == 0, f"{n} rows differ")

n = int((df["shipping_date"] < df["order_date"]).sum())
check("shipping date not before order date", n == 0, f"{n} rows")

days_calc = (df["shipping_date"].dt.normalize() - df["order_date"].dt.normalize()).dt.days
n = int((days_calc != df["days_for_shipping_real"]).sum())
check("shipping_date - order_date = days_for_shipping_real", n == 0, f"{n} rows differ")

n = int((df["order_item_quantity"] <= 0).sum())
check("quantity > 0", n == 0, f"{n} rows")

n = int(((df["sales"] - df["order_item_product_price"] * df["order_item_quantity"]).abs() > 0.01).sum())
check("sales = price x quantity", n == 0, f"{n} rows differ")

pd.DataFrame(results)

,check,result,detail
0,order_item_id is unique,PASS,
1,no nulls in key columns,PASS,0 nulls
2,late_delivery_risk = real > scheduled (non-can...,PASS,0 rows differ
3,cancelled orders have late_delivery_risk = 0,PASS,0 rows
4,'Shipping canceled' = order CANCELED or SUSPEC...,PASS,0 rows differ
5,shipping date not before order date,PASS,0 rows
6,shipping_date - order_date = days_for_shipping...,PASS,0 rows differ
7,quantity > 0,PASS,0 rows
8,sales = price x quantity,PASS,0 rows differ


In [5]:
df.groupby(df["order_date"].dt.to_period("M")).size().tail(8)

order_date
2017-06    4951
2017-07    5318
2017-08    5305
2017-09    5189
2017-10    2255
2017-11    2055
2017-12    2124
2018-01    2123
Freq: M, dtype: int64

In [6]:
mask = (df["days_for_shipping_real"] > df["days_for_shipment_scheduled"]).astype(int) != df["late_delivery_risk"]
print(pd.crosstab(df.loc[mask, "delivery_status"], df.loc[mask, "late_delivery_risk"]))
df.loc[mask, ["days_for_shipping_real", "days_for_shipment_scheduled", "late_delivery_risk",
              "delivery_status", "order_status"]].head(10)

late_delivery_risk     0
delivery_status         
Shipping canceled   4423


,days_for_shipping_real,days_for_shipment_scheduled,late_delivery_risk,delivery_status,order_status
5,6,4,0,Shipping canceled,CANCELED
10,6,2,0,Shipping canceled,SUSPECTED_FRAUD
23,3,2,0,Shipping canceled,CANCELED
39,1,0,0,Shipping canceled,CANCELED
183,5,4,0,Shipping canceled,SUSPECTED_FRAUD
184,5,4,0,Shipping canceled,SUSPECTED_FRAUD
185,6,4,0,Shipping canceled,SUSPECTED_FRAUD
186,5,4,0,Shipping canceled,SUSPECTED_FRAUD
192,6,4,0,Shipping canceled,SUSPECTED_FRAUD
193,5,4,0,Shipping canceled,SUSPECTED_FRAUD


In [7]:
month = df["order_date"].dt.to_period("M")
(df.assign(month=month)
   .groupby("month")
   .agg(rows=("order_item_id", "size"),
        orders=("order_id", "nunique"),
        products=("product_card_id", "nunique"),
        categories=("category_name", "nunique"),
        markets=("market", "nunique"))
   .tail(8))

,rows,orders,products,categories,markets
month,,,,,
2017-06,4951,1676,42,24,2
2017-07,5318,1776,42,24,1
2017-08,5305,1768,42,24,1
2017-09,5189,1723,53,26,1
2017-10,2255,2101,20,20,1
2017-11,2055,2055,8,8,2
2017-12,2124,2124,14,14,1
2018-01,2123,2123,10,10,1


In [8]:
df["is_late"] = (~df["is_cancelled"]) & (df["days_for_shipping_real"] > df["days_for_shipment_scheduled"])
df["in_analysis_period"] = df["order_date"] < "2017-10-01"

In [9]:
n = int((((df["days_for_shipping_real"] > df["days_for_shipment_scheduled"]).astype(int) != df["late_delivery_risk"]) & ~df["is_cancelled"]).sum())
check("late_delivery_risk = real > scheduled (non-cancelled orders)", n == 0, f"{n} rows differ")

n = int((df.loc[df["is_cancelled"], "late_delivery_risk"] != 0).sum())
check("cancelled orders have late_delivery_risk = 0", n == 0, f"{n} rows")

In [10]:
pd.crosstab(df["order_date"].dt.year, df["market"])

market,Africa,Europe,LATAM,Pacific Asia,USCA
order_date,,,,,
2015,0,24914,25775,11961,0
2016,10356,3872,0,22641,25681
2017,1258,21466,25819,4535,118
2018,0,0,0,2123,0


In [11]:
import os
os.makedirs("data-processed", exist_ok=True)
df.to_csv("data-processed/order_items_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (180519, 46)
